In [1]:
from nmm.utils.generators import GKLS
from nmm import Qobj
from qutip import Qobj as Qobjq
from nmm.redfield import re_jax
import jax.numpy as jnp
import jax
jax.config.update("jax_enable_x64", True)


In [2]:
H = 15  * Qobjq(jnp.array([[1,0],[0,-1]]))/2 + 9 *Qobjq(jnp.array([[0,1],[1,0]]))
Q = Qobjq(jnp.array([[0,1],[1,0]]))
Hjax = 15  * Qobj(jnp.array([[1,0],[0,-1]]))/2 + 9 *Qobj(jnp.array([[0,1],[1,0]]))
Qjax = Qobj(jnp.array([[0,1],[1,0]]))
t = None

An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.


In [3]:
Hjax.data

Array([[ 7.5,  9. ],
       [ 9. , -7.5]], dtype=float64)

In [4]:
py = GKLS(Hjax, Qjax , t)
py.jump_operators(Qjax)

{23.43074902772: Operator: 
  [[-0.24590164 -0.11517417]
  [ 0.52501023  0.24590164]],
 -23.43074902772: Operator: 
  [[-0.24590164  0.52501023]
  [-0.11517417  0.24590164]],
 0: Operator: 
  [[ 0.49180328  0.59016393]
  [ 0.59016393 -0.49180328]]}

In [4]:
import numpy as np 
from qutip import OhmicEnvironment
tfit=np.linspace(0,50,200)
bathr=OhmicEnvironment(T=0.5,alpha=0.05*np.pi,wc=1,s=3/2)
envfit2,fitinfo=bathr.approximate("espira-I",tlist=tfit,Nr=4)
print(fitinfo["summary"])

Result of fitting Correlation Function with 4 terms: 
 
 Parameters|   ckr    |   cki    |   vkr    |  vki 
 1         | 4.36e-02 | 8.03e-02 | 2.65e+00 |2.66e+00
 2         | 4.03e-02 |-8.01e-02 | 1.45e+00 |3.58e-01
 3         |-2.56e-03 | 1.43e-03 | 1.97e-01 |-8.11e-02
 4         |-7.09e-04 | 4.26e-04 | 4.97e-02 |-1.33e-02
 
A RMSE of  2.69e-03-2.99e-04j was obtained for the Correlation Function.
The current fit took  0.615186 seconds.


In [6]:
pyj = re_jax.JAXGKLS(Hjax, t,[envfit2],[Qjax],picture="I",LS=True)
ws,d=pyj.jump_operators(Qjax)
d

Array([[[-0.24590164,  0.52501023],
        [-0.11517417,  0.24590164]],

       [[ 0.49180328,  0.59016393],
        [ 0.59016393, -0.49180328]],

       [[-0.24590164, -0.11517417],
        [ 0.52501023,  0.24590164]],

       [[ 0.        ,  0.        ],
        [ 0.        ,  0.        ]]], dtype=float64)

In [7]:
tensort = pyj.generator_jax(30)

In [8]:
tensort

Array([[ 0.00000000e+00+0.j        ,  5.06240736e-03+0.00282579j,
         5.06240736e-03-0.00282579j,  2.16840434e-19+0.j        ],
       [-5.06240736e-03+0.00282579j,  6.50521303e-19+0.00227473j,
        -4.33680869e-19+0.j        ,  5.06240736e-03-0.00282579j],
       [-5.06240736e-03-0.00282579j,  0.00000000e+00+0.j        ,
        -8.67361738e-19-0.00227473j,  5.06240736e-03+0.00282579j],
       [ 0.00000000e+00+0.j        , -5.06240736e-03-0.00282579j,
        -5.06240736e-03+0.00282579j, -2.16840434e-19+0.j        ]],      dtype=complex128)

In [12]:
red2.generator(30).full()

array([[ 0.        +0.j        ,  0.00506241+0.00282579j,
         0.00506241-0.00282579j,  0.        +0.j        ],
       [-0.00506241+0.00282579j,  0.        +0.00227473j,
         0.        +0.j        ,  0.00506241-0.00282579j],
       [-0.00506241-0.00282579j,  0.        +0.j        ,
         0.        -0.00227473j,  0.00506241+0.00282579j],
       [ 0.        +0.j        , -0.00506241-0.00282579j,
        -0.00506241+0.00282579j,  0.        +0.j        ]])

In [5]:
from nmm import redfield

In [4]:
red2=redfield.redfield(Hsys=H,t=t,baths=[envfit2],Qs=[Q],eps=1e-8,matsubara=True,ls=True,picture="I")
#result_red2=red2.evolution(rho0,method="Tsit5")

NameError: name 'redfield' is not defined

In [ ]:
result = pyj.evolution(Qobj(jnp.array([[1,0],[0,0]],dtype=jnp.complex128)),10,0.01,jnp.linspace(0.0, 10.0, 100))

/home/gerardo/.pyenv/versions/qutip-dev/lib/python3.12/site-packages/equinox/_jit.py:55: UserWarning: Complex dtype support in Diffrax is a work in progress and may not yet produce correct results. Consider splitting your computation into real and imaginary parts instead.
  out = fun(*args, **kwargs)


In [ ]:
result

(Array([ 0.        ,  0.1010101 ,  0.2020202 ,  0.3030303 ,  0.4040404 ,
         0.50505051,  0.60606061,  0.70707071,  0.80808081,  0.90909091,
         1.01010101,  1.11111111,  1.21212121,  1.31313131,  1.41414141,
         1.51515152,  1.61616162,  1.71717172,  1.81818182,  1.91919192,
         2.02020202,  2.12121212,  2.22222222,  2.32323232,  2.42424242,
         2.52525253,  2.62626263,  2.72727273,  2.82828283,  2.92929293,
         3.03030303,  3.13131313,  3.23232323,  3.33333333,  3.43434343,
         3.53535354,  3.63636364,  3.73737374,  3.83838384,  3.93939394,
         4.04040404,  4.14141414,  4.24242424,  4.34343434,  4.44444444,
         4.54545455,  4.64646465,  4.74747475,  4.84848485,  4.94949495,
         5.05050505,  5.15151515,  5.25252525,  5.35353535,  5.45454545,
         5.55555556,  5.65656566,  5.75757576,  5.85858586,  5.95959596,
         6.06060606,  6.16161616,  6.26262626,  6.36363636,  6.46464646,
         6.56565657,  6.66666667,  6.76767677,  6.8

### Beyond a Qubit

In [3]:
import qutip as qt
import numpy as np
from qutip import UnderDampedEnvironment
# Qubit parameters
w = 1
delta1=0.1
delta2=0.2
# System operators
H1   = w / 2 * qt.tensor(qt.sigmaz(), qt.identity(2),qt.identity(2))
H2   = (w + w*delta1) / 2 * qt.tensor(qt.identity(2), qt.sigmaz(), qt.identity(2))
H3   = (w - w*delta2) / 2 * qt.tensor(qt.identity(2),qt.identity(2), qt.sigmaz())
H12  = lambda J12 : J12 * (qt.tensor(qt.sigmap(), qt.sigmam(),qt.identity(2)) + qt.tensor(qt.sigmam(), qt.sigmap(),qt.identity(2)))
H23  = lambda J23 : J23 * (qt.tensor(qt.identity(2),qt.sigmap(), qt.sigmam()) + qt.tensor(qt.identity(2),qt.sigmam(), qt.sigmap()))
H13  = lambda J13 : J13 * (qt.tensor(qt.sigmap(),qt.identity(2), qt.sigmam()) + qt.tensor(qt.sigmam(),qt.identity(2), qt.sigmap()))
Hsys = lambda J12 : H1 + H2+H3 + H12(J12)+ H13(J12)+ H23(J12)

# Cutoff frequencies
gamma = 1

# Temperatures
Tc = 1#Tbar - Delta_T
Tm = 0.8#Tbar + Delta_T
Th= 1.2
# Coupling operators
Q1 = qt.tensor(qt.sigmax(), qt.identity(2), qt.identity(2))
Q2 = qt.tensor(qt.identity(2), qt.sigmax(), qt.identity(2))
Q3 = qt.tensor(qt.identity(2), qt.identity(2), qt.sigmax())
print(Tc,Tm)

1 0.8


In [4]:
# qubit-qubit and qubit-bath coupling strengths
g = 0.3 #qubit-qubit
lam1 = 0.45 #bath1
lam2 = 0.55#bath2
lam3 = 0.65 #bath3
# choose arbitrary initial state
ket=qt.ket("000")#
rho=ket*ket.dag()
rho0 =rho#qt.Qobj([[0.3,0,0,0],[0,0.2,0,0.1],[0,0,0.5,0],[0,0.1,0,0]])
rho0.dims=rho.dims
# simulation time span
tlist = np.linspace(0, 50, 200)
H=Hsys(g)

In [5]:
ws=np.linspace(0,80,2000)
w2=np.linspace(-80,80,2000)
t=np.linspace(0,40,5000)

In [6]:
bath1=UnderDampedEnvironment(T=Tc,lam=lam1,gamma=gamma,w0=1)
bath1approx=bath1.approx_by_matsubara(Nk=500)
bath1corr,finfo=bath1.approx_by_cf_fit(t,tag="bath 1",Ni_max=1,Nr_max=3,full_ansatz=True)
print(finfo['summary'])

/home/gerardo/Documents/gsuarezr/qutip_gsoc_app/qutip/core/environment.py:1485: FutureWarning: The API has changed. Please use approximate("matsubara", ...) instead of approx_by_matsubara(...).
  warnings.warn(
/home/gerardo/Documents/gsuarezr/qutip_gsoc_app/qutip/core/environment.py:719: FutureWarning: The API has changed. Please use approximate("cf", ...) instead of approx_by_cf_fit(...).
  warnings.warn('The API has changed. Please use approximate("cf", ...)'


Correlation function fit:

Result of fitting the real part of                        |Result of fitting the imaginary part                       
the correlation function with 2 terms:                    |of the correlation function with 1 terms:                  
                                                          |                                                           
 Parameters|   ckr    |   vkr    |   vki    |  cki        | Parameters|   ckr    |   vkr    |   vki    |  cki         
 1         |-1.86e-03 |-7.37e+00 | 4.96e-07 |4.60e-01     | 1         |-1.17e-01 |-5.00e-01 | 8.66e-01 |1.07e-05      
 2         | 2.19e-01 |-5.00e-01 | 8.66e-01 |-1.08e-01    |                                                           
                                                          |A RMSE of  3.82e-05 was obtained for the the imaginary part
A RMSE of  8.49e-06 was obtained for the the real part of |of the correlation function.                               
the correlation funct

In [7]:
bath2=UnderDampedEnvironment(T=Tm,lam=lam2,gamma=gamma,w0=1)
bath2approx=bath2.approx_by_matsubara(Nk=6)
bath2corr,finfo=bath2.approx_by_cf_fit(t,Ni_max=1,Nr_max=3,tag="bath 2",full_ansatz=True)
print(finfo['summary'])

Correlation function fit:

Result of fitting the real part of                        |Result of fitting the imaginary part                       
the correlation function with 2 terms:                    |of the correlation function with 1 terms:                  
                                                          |                                                           
 Parameters|   ckr    |   vkr    |   vki    |  cki        | Parameters|   ckr    |   vkr    |   vki    |  cki         
 1         | 2.73e-01 |-5.00e-01 | 8.66e-01 |-1.22e-01    | 1         | 1.75e-01 |-5.00e-01 |-8.66e-01 |1.59e-05      
 2         |-4.24e-03 |-5.88e+00 |-1.70e-06 |-8.69e-02    |                                                           
                                                          |A RMSE of  3.82e-05 was obtained for the the imaginary part
A RMSE of  1.42e-05 was obtained for the the real part of |of the correlation function.                               
the correlation funct

In [8]:
bath3=UnderDampedEnvironment(T=Th,lam=lam3,gamma=gamma,w0=1)
bath3approx=bath3.approx_by_matsubara(Nk=500)
bath3corr,finfo=bath3.approx_by_cf_fit(t,Ni_max=1,Nr_max=3,tag="bath 3",full_ansatz=True)
print(finfo['summary'])

Correlation function fit:

Result of fitting the real part of                        |Result of fitting the imaginary part                       
the correlation function with 2 terms:                    |of the correlation function with 1 terms:                  
                                                          |                                                           
 Parameters|   ckr    |   vkr    |   vki    |  cki        | Parameters|   ckr    |   vkr    |   vki    |  cki         
 1         | 5.36e-01 |-5.00e-01 |-8.66e-01 |2.76e-01     | 1         | 2.44e-01 |-5.00e-01 |-8.66e-01 |2.22e-05      
 2         |-2.75e-03 |-8.97e+00 |-7.70e-07 |-1.91e-01    |                                                           
                                                          |A RMSE of  3.82e-05 was obtained for the the imaginary part
A RMSE of  5.49e-06 was obtained for the the real part of |of the correlation function.                               
the correlation funct

In [9]:
from qutip.solver.heom import BosonicBath,HEOMSolver,BathExponent
import matplotlib.pyplot as plt
options = {'nsteps':15000, 'store_states':True,'store_ados':True,'rtol':1e-12,'atol':1e-12}

b1heom=BosonicBath.from_environment(bath1corr,Q1)
b2heom=BosonicBath.from_environment(bath2corr,Q2)
b3heom=BosonicBath.from_environment(bath3corr,Q3)

In [10]:
rd=re_jax.JAXGKLS(Hsys=Qobj(H.full()),baths=[bath1corr],t=jnp.array(tlist),Qs=[Qobj(Q1.full())],picture="S",LS=True)


An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.


In [11]:
Qobjq(rd.generator_jax(4))

Quantum object: dims=[[64], [64]], shape=(64, 64), type='oper', dtype=Dense, isherm=False
Qobj data =
[[ 0.00000000e+00+0.j          0.00000000e+00+0.j
   0.00000000e+00+0.j         ...  0.00000000e+00+0.j
   0.00000000e+00+0.j          0.00000000e+00+0.j        ]
 [ 0.00000000e+00+0.j         -3.46944695e-18-0.81049177j
   7.87760359e-03+0.27595703j ...  0.00000000e+00+0.j
   0.00000000e+00+0.j          0.00000000e+00+0.j        ]
 [ 0.00000000e+00+0.j         -7.87760359e-03+0.27595703j
   1.73472348e-18-1.10325934j ...  0.00000000e+00+0.j
   0.00000000e+00+0.j          0.00000000e+00+0.j        ]
 ...
 [ 0.00000000e+00+0.j          0.00000000e+00+0.j
   0.00000000e+00+0.j         ...  3.46944695e-18+1.09206281j
  -1.87120513e-03+0.31139336j  0.00000000e+00+0.j        ]
 [ 0.00000000e+00+0.j          0.00000000e+00+0.j
   0.00000000e+00+0.j         ...  1.87120513e-03+0.31139336j
  -1.73472348e-18+0.79041373j  0.00000000e+00+0.j        ]
 [ 0.00000000e+00+0.j          0.00000000e+00+

In [14]:
red2=redfield.redfield(Hsys=H,t=t,baths=[bath1corr,bath2corr,bath3corr],Qs=[Q1,Q2,Q3],eps=1e-8,matsubara=True,ls=True,picture="S")
red2.generator(4)

Quantum object: dims=[[[2, 2, 2], [2, 2, 2]], [[2, 2, 2], [2, 2, 2]]], shape=(64, 64), type='super', dtype=CSR, isherm=False
Qobj data =
[[ 0.        +0.j          0.        +0.j          0.        +0.j
  ...  0.        +0.j          0.        +0.j
   0.        +0.j        ]
 [ 0.        +0.j          0.        -1.41579975j -0.01555836+0.4307954j
  ...  0.        +0.j          0.        +0.j
   0.        +0.j        ]
 [ 0.        +0.j          0.01555836+0.4307954j   0.        -1.52130569j
  ...  0.        +0.j          0.        +0.j
   0.        +0.j        ]
 ...
 [ 0.        +0.j          0.        +0.j          0.        +0.j
  ...  0.        +1.41372169j -0.0648426 +0.43379739j
   0.        +0.j        ]
 [ 0.        +0.j          0.        +0.j          0.        +0.j
  ...  0.0648426 +0.43379739j  0.        +1.26274254j
   0.        +0.j        ]
 [ 0.        +0.j          0.        +0.j          0.        +0.j
  ...  0.        +0.j          0.        +0.j
   0.        +0.j   

In [5]:
from nmm.redfield.re_jax import JAXCUM

In [6]:
cum  =  JAXCUM(Hjax, t,[envfit2],[Qjax],picture="I",LS=True)

In [7]:
res= cum.evolution(rho0=jnp.array([[1,0],[0,0]],dtype=jnp.complex128),t=jnp.linspace(0,100,2))
res

Array([[[1.        +0.00000000e+00j, 0.        +0.00000000e+00j],
        [0.        +0.00000000e+00j, 0.        +0.00000000e+00j]],

       [[0.72951424-2.00421302e-18j, 0.23112183-2.36005012e-03j],
        [0.23112183+2.36005012e-03j, 0.27048576+1.09352958e-18j]]],      dtype=complex128)

In [8]:
import qutip as qt

In [9]:
rho0 = qt.Qobj(jnp.array([[1,0],[0,0]],dtype=jnp.complex128))
t=np.array(jnp.linspace(10,100,2))

In [10]:
from nmm import csolve

In [11]:
cum=csolve(Hsys=H,t=t,baths=[envfit2],Qs=[Q],eps=1e-4,cython=False,matsubara=True,ls=True)
result_cum=cum.evolution(rho0)
result_cum

Calculating time dependent generators: 100%|██████████| 9/9 [00:00<00:00, 14896.90it/s]
Computing Exponential of Generators . . . .: 2it [00:00, 3869.28it/s]


[Quantum object: dims=[[2], [2]], shape=(2, 2), type='oper', dtype=Dense, isherm=True
 Qobj data =
 [[0.86840215+1.08420217e-19j 0.10990602-1.86638865e-03j]
  [0.10990602+1.86638865e-03j 0.13159785-1.92276977e-19j]],
 Quantum object: dims=[[2], [2]], shape=(2, 2), type='oper', dtype=Dense, isherm=True
 Qobj data =
 [[0.72935005-3.46944695e-18j 0.23129987-4.27216462e-03j]
  [0.23129987+4.27216462e-03j 0.27064995+2.40703942e-18j]]]